In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

In [3]:


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = MNIST(root="./data", train=True, download=True, transform=transform)
testset = MNIST(root="./data", train=False, download=True, transform=transform)

In [4]:
print("Training samples:", len(trainset))
print("Testing samples:", len(testset))

Training samples: 60000
Testing samples: 10000


In [13]:
image, label = trainset[0]
print(image.shape)

torch.Size([1, 28, 28])


In [5]:
train_dataloader=DataLoader(trainset,batch_size=64,shuffle=True)
test_dataloader=DataLoader(testset,batch_size=64,shuffle=True)

In [19]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=self.pool(x)
        x=x.view(x.size(0),-1)
        x = self.fc_layers(x)

        return x

In [20]:
model=CNN()

In [21]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [22]:
epochs = 50

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in train_dataloader:
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate loss
        epoch_training_loss += loss.item()

    # Average loss for the epoch
    epoch_training_loss /= len(train_dataloader)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_training_loss:.4f}")

Epoch [1/50], Loss: 0.1650
Epoch [2/50], Loss: 0.0447
Epoch [3/50], Loss: 0.0312
Epoch [4/50], Loss: 0.0239
Epoch [5/50], Loss: 0.0195
Epoch [6/50], Loss: 0.0173
Epoch [7/50], Loss: 0.0156
Epoch [8/50], Loss: 0.0120
Epoch [9/50], Loss: 0.0115
Epoch [10/50], Loss: 0.0095
Epoch [11/50], Loss: 0.0081
Epoch [12/50], Loss: 0.0083
Epoch [13/50], Loss: 0.0093
Epoch [14/50], Loss: 0.0060
Epoch [15/50], Loss: 0.0066
Epoch [16/50], Loss: 0.0062
Epoch [17/50], Loss: 0.0065
Epoch [18/50], Loss: 0.0061
Epoch [19/50], Loss: 0.0042
Epoch [20/50], Loss: 0.0062
Epoch [21/50], Loss: 0.0050
Epoch [22/50], Loss: 0.0046
Epoch [23/50], Loss: 0.0065
Epoch [24/50], Loss: 0.0056
Epoch [25/50], Loss: 0.0036
Epoch [26/50], Loss: 0.0036
Epoch [27/50], Loss: 0.0036
Epoch [28/50], Loss: 0.0051
Epoch [29/50], Loss: 0.0048
Epoch [30/50], Loss: 0.0042
Epoch [31/50], Loss: 0.0038
Epoch [32/50], Loss: 0.0052
Epoch [33/50], Loss: 0.0030
Epoch [34/50], Loss: 0.0035
Epoch [35/50], Loss: 0.0036
Epoch [36/50], Loss: 0.0054
E

In [23]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_dataloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 99.35000000000001


In [27]:
torch.save(model.state_dict(), "mnist_cnn.pth")
print("Model saved successfully!")

Model saved successfully!
